In [5]:
"""
Real-World Stress-Test Series #01 — Financial & Credit Risk
Dataset: Taiwanese Company Bankruptcy Prediction (Kaggle)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ==========================================================
# 1. LOAD & CLEAN DATA
# ==========================================================
CSV_PATH = "data.csv"  # adjust path if needed after upload

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()  # strip leading/trailing whitespace

print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
print(df['Bankrupt?'].value_counts())
print(df['Bankrupt?'].value_counts(normalize=True).round(4))

# ==========================================================
# 2. TRAIN-TEST SPLIT (stratified, no leakage)
# ==========================================================
X = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Bankrupt firms in test set: {y_test.sum()} out of {len(y_test)}")

# ==========================================================
# 3. BASELINE MODEL (trained ONCE, never retrained)
# ==========================================================
model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

# ==========================================================
# 4. STRESS LADDER: identify the four target columns
# ==========================================================
# Deterioration logic — documented, not arbitrary:
#
#   Variable                | Direction | Why
#   -------------------------|-----------|----------------------------------
#   ROA(C)                   |    v      | Lower profitability = weaker
#                             |           | financial performance
#   Cash/Total Assets        |    v      | Lower liquidity = reduced
#                             |           | ability to absorb a shock
#   Debt ratio %              |    ^      | Higher leverage = greater
#                             |           | financial burden
#   Borrowing dependency      |    ^      | Greater reliance on borrowing =
#                             |           | higher financial pressure
#
# "10% stress" means a 10% RELATIVE change applied to these four ratios
# specifically -- not a claim that the company's overall financial
# condition deteriorated by 10%.

roa_col        = [c for c in X.columns if 'ROA(C)' in c][0]
cash_col       = [c for c in X.columns if c == 'Cash/Total Assets'][0]
debt_col       = [c for c in X.columns if c == 'Debt ratio %'][0]
borrow_col     = [c for c in X.columns if c == 'Borrowing dependency'][0]

print("\nColumns being stressed (verifying ranges rather than assuming them):")
for c in [roa_col, cash_col, debt_col, borrow_col]:
    print(f"  {c}: min={X_test[c].min():.4f}  max={X_test[c].max():.4f}  "
          f"mean={X_test[c].mean():.4f}  median={X_test[c].median():.4f}")

def apply_stress(X_input, pct):
    """Return a stressed COPY of X_input. pct is a fraction, e.g. 0.10 for 10%.
    Clipping to [0,1] is a safety bound, not the main mechanism -- checked
    below to confirm how often it actually triggers."""
    X_stressed = X_input.copy()
    X_stressed[roa_col]    = np.clip(X_stressed[roa_col]    * (1 - pct), 0, 1)
    X_stressed[cash_col]   = np.clip(X_stressed[cash_col]   * (1 - pct), 0, 1)
    X_stressed[debt_col]   = np.clip(X_stressed[debt_col]   * (1 + pct), 0, 1)
    X_stressed[borrow_col] = np.clip(X_stressed[borrow_col] * (1 + pct), 0, 1)
    return X_stressed

def check_clip_impact(X_input, pct):
    """Report how many rows would exceed [0,1] BEFORE clipping is applied,
    so clipping's effect on the experiment is measured, not assumed."""
    debt_raw = X_input[debt_col] * (1 + pct)
    borrow_raw = X_input[borrow_col] * (1 + pct)
    n_debt = (debt_raw > 1.0).sum()
    n_borrow = (borrow_raw > 1.0).sum()
    total = len(X_input)
    print(f"  {int(pct*100)}% stress: Debt ratio clipped in {n_debt}/{total} rows "
          f"({n_debt/total:.2%}), Borrowing dependency clipped in {n_borrow}/{total} rows "
          f"({n_borrow/total:.2%})")

# ==========================================================
# 5. RUN THE LADDER: 0%, 10%, 20%, 30%
# ==========================================================
print("\nClipping impact check (how often the [0,1] safety bound actually fires):")
for pct in [0.10, 0.20, 0.30]:
    check_clip_impact(X_test, pct)

stress_levels = [0.0, 0.10, 0.20, 0.30]
results = []

for pct in stress_levels:
    X_eval = apply_stress(X_test, pct) if pct > 0 else X_test.copy()
    preds = model.predict(X_eval)          # SAME model every time, never retrained
    probs = model.predict_proba(X_eval)[:, 1]  # predicted bankruptcy probability

    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    f1   = f1_score(y_test, preds, zero_division=0)
    cm   = confusion_matrix(y_test, preds)
    fn   = cm[1][0]  # actual=1, predicted=0

    results.append({
        'stress_pct': int(pct * 100),
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'false_negatives': fn,
        'confusion_matrix': cm,
        'preds': preds,
        'mean_prob': probs.mean(),
        'mean_prob_bankrupt_firms': probs[y_test.values == 1].mean(),
    })

results_df = pd.DataFrame(results).drop(columns=['confusion_matrix', 'preds'])

# ==========================================================
# 6a. PREDICTION-CHANGE & PROBABILITY-SHIFT CHECK
# ==========================================================
# Class predictions can stay identical while the underlying risk score
# still moves -- this checks both, rather than assuming "no class change"
# means "nothing changed."
baseline_preds = results[0]['preds']
print("\n" + "=" * 72)
print(" DID INDIVIDUAL PREDICTIONS OR RISK SCORES ACTUALLY MOVE?")
print("=" * 72)
for r in results:
    n_changed = int(np.sum(baseline_preds != r['preds']))
    print(f"  {r['stress_pct']:>3}% stress: {n_changed}/{len(y_test)} predicted CLASSES changed | "
          f"mean predicted bankruptcy probability = {r['mean_prob']:.4f} | "
          f"mean probability among actually-bankrupt firms = {r['mean_prob_bankrupt_firms']:.4f}")

# ==========================================================
# 6. ASCII COMPARISON TABLE
# ==========================================================
print("\n" + "=" * 72)
print(" STRESS LADDER RESULTS — SAME MODEL, INCREASING STRESS")
print("=" * 72)
header = f"{'Stress %':>10} | {'Accuracy':>9} | {'Precision':>10} | {'Recall':>8} | {'F1':>7} | {'FN':>4}"
print(header)
print("-" * len(header))
for r in results:
    print(f"{r['stress_pct']:>9}% | {r['accuracy']:>9.1%} | {r['precision']:>10.1%} | "
          f"{r['recall']:>8.1%} | {r['f1']:>7.1%} | {r['false_negatives']:>4}")
print("=" * 72)

baseline_recall = results[0]['recall']
severe_recall   = results[-1]['recall']
print(f"\nRecall change from 0% to 30% stress: "
      f"{(severe_recall - baseline_recall) * 100:+.1f} percentage points")

# ==========================================================
# 7. VISUAL 1 — Recall stress ladder line chart
# ==========================================================
plt.figure(figsize=(7, 5))
plt.grid(True, alpha=0.3)
recalls = [r['recall'] * 100 for r in results]
levels = [r['stress_pct'] for r in results]
plt.plot(levels, recalls, marker='o', linewidth=2.5, markersize=9, color='#c0392b')
for x, y_val in zip(levels, recalls):
    plt.annotate(f"{y_val:.1f}%", (x, y_val), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=10, fontweight='bold')
plt.xlabel("Simulated Stress Level (%)", fontsize=11)
plt.ylabel("Bankruptcy Recall (%)", fontsize=11)
plt.title("Model Recall Under Increasing Simulated Financial Stress", fontsize=12, fontweight='bold')
plt.xticks(levels, [f"{l}%" for l in levels])
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig("recall_stress_ladder.png", dpi=200)
plt.close()
print("\nSaved: recall_stress_ladder.png")

# ==========================================================
# 8. VISUAL 2 — Confusion matrices: 0% vs 30%
# ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
cm_baseline = results[0]['confusion_matrix']
cm_severe   = results[-1]['confusion_matrix']

labels = ['Pred: Solvent', 'Pred: Bankrupt']
ylabels = ['Actual: Solvent', 'Actual: Bankrupt']

for ax, cm, title in zip(
    axes,
    [cm_baseline, cm_severe],
    ["Baseline (0% Stress)", "30% Simulated Stress"]
):
    im = ax.imshow(cm, cmap='Reds')
    ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
    ax.set_yticks([0, 1]); ax.set_yticklabels(ylabels)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=13, fontweight='bold',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("confusion_matrix_comparison.png", dpi=200)
plt.close()
print("Saved: confusion_matrix_comparison.png")

# ==========================================================
# 8b. VISUAL 3 — Mean predicted bankruptcy probability by stress level
# ==========================================================
plt.figure(figsize=(7, 5))
plt.grid(True, alpha=0.3)
mean_probs = [r['mean_prob_bankrupt_firms'] * 100 for r in results]
plt.plot(levels, mean_probs, marker='s', linewidth=2.5, markersize=9, color='#2c3e50')
for x, y_val in zip(levels, mean_probs):
    plt.annotate(f"{y_val:.1f}%", (x, y_val), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=10, fontweight='bold')
plt.axhline(50, color='gray', linestyle='--', linewidth=1, label='Classification threshold (50%)')
plt.xlabel("Simulated Stress Level (%)", fontsize=11)
plt.ylabel("Mean Predicted Bankruptcy Probability (%)", fontsize=11)
plt.title("Risk Score Movement Among Actually-Bankrupt Firms", fontsize=12, fontweight='bold')
plt.xticks(levels, [f"{l}%" for l in levels])
plt.legend()
plt.tight_layout()
plt.savefig("probability_shift.png", dpi=200)
plt.close()
print("Saved: probability_shift.png")

# ==========================================================
# 9. RESULTS TABLE FOR README
# ==========================================================
print("\n" + "=" * 72)
print(" COPY THIS TABLE INTO YOUR README / POST")
print("=" * 72)
print(results_df.to_string(index=False))


Dataset shape: (6819, 96)
Missing values: 0
Duplicate rows: 0

Target distribution:
Bankrupt?
0    6599
1     220
Name: count, dtype: int64
Bankrupt?
0    0.9677
1    0.0323
Name: proportion, dtype: float64

Train size: 5455 | Test size: 1364
Bankrupt firms in test set: 44 out of 1364

Columns being stressed (verifying ranges rather than assuming them):
  ROA(C) before interest and depreciation before interest: min=0.0000  max=0.8180  mean=0.5081  median=0.5061
  Cash/Total Assets: min=0.0002  max=0.8918  mean=0.1232  median=0.0777
  Debt ratio %: min=0.0004  max=0.3385  mean=0.1129  median=0.1117
  Borrowing dependency: min=0.0000  max=1.0000  mean=0.3746  median=0.3726

Clipping impact check (how often the [0,1] safety bound actually fires):
  10% stress: Debt ratio clipped in 0/1364 rows (0.00%), Borrowing dependency clipped in 1/1364 rows (0.07%)
  20% stress: Debt ratio clipped in 0/1364 rows (0.00%), Borrowing dependency clipped in 1/1364 rows (0.07%)
  30% stress: Debt ratio cli